In [7]:
# Core scientific Python stack plus geospatial/network utilities for basin exploration.

import os
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx
import py3dep
from pynhd import NLDI
import rioxarray
from pyproj import Transformer
from shapely.ops import transform
from pysheds.grid import Grid
import geopandas as gpd
import pandas as pd
from pynhd import HP3D, NLDI, GeoConnex, NHDPlusHR, WaterData
import pygeohydro as gh
from pygeohydro import NWIS, plot, WBD
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

import requests
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import io

from pygeohydro import get_us_states

%matplotlib ipympl

In [2]:
# Read the list of USGS sites from Station_List.csv in the Outputs folder.
# The CSV must have a column named 'site_no' (or update site_col to match your heading).

from pathlib import Path

# Paths
path_data    = "Outputs"
path_figures = "Figures"
path_usgs_out = os.path.join(path_data, "USGS_ins_data")
if not os.path.exists(path_usgs_out):
    os.makedirs(path_usgs_out)

# Create folders if they do not already exist.
for folder in [path_data, path_figures, path_usgs_out]:
    if not os.path.exists(folder):
        os.makedirs(folder)

# Read station list — update 'site_no' to match the exact column heading in your CSV.
station_file = os.path.join(path_data, "site_info.csv")

if not os.path.exists(station_file):
    raise FileNotFoundError(
        f"Could not find {station_file}\n"
        f"Current working directory: {os.getcwd()}\n"
        f"Files here: {os.listdir('.')}"
    )

stations_df = pd.read_csv(station_file, dtype=str)
site_col    = stations_df.columns[0]   # auto-detects first column as site ID
# Pad USGS site IDs back to 8 digits with leading zeros
stations_df[site_col] = stations_df[site_col].str.strip().str.zfill(8)
print(f"Using column '{site_col}' as USGS site IDs")

USGS_sites = stations_df[site_col].str.strip().tolist()
print(f"Total stations to process: {len(USGS_sites)}")
print(USGS_sites)

Using column 'site_no' as USGS site IDs
Total stations to process: 101
['07148400', '07152000', '07153000', '07157950', '07158000', '07159100', '07159750', '07160000', '07160350', '07160500', '07161450', '07164600', '07165562', '07165565', '07165570', '07171000', '07174400', '07175500', '07176000', '07176500', '07177650', '07177800', '07185000', '07188000', '07189540', '07189542', '07191000', '07191220', '07191500', '07195855', '07195865', '07196000', '07196500', '07197000', '07197360', '07228500', '07229050', '07229200', '07229300', '07230000', '07230500', '07231000', '07231500', '07234000', '07237500', '07238000', '07239300', '07239450', '07239500', '07239700', '07240000', '07241000', '07241520', '07241550', '07242000', '07242380', '07243500', '07245000', '07247015', '07247250', '07247500', '07249413', '07249455', '07249800', '07249985', '07300500', '07301110', '07301420', '07303000', '07305000', '07307028', '07311000', '07311500', '07315700', '07316500', '07324200', '07324400', '073

In [3]:
nwis = NWIS()

In [ ]:
start_date = "2000-01-01"
end_date   = "2025-12-31"

for USGSsite in USGS_sites:

    out_file = os.path.join(path_usgs_out, f"{USGSsite}_USGS_inst.csv")

    if os.path.exists(out_file):
        print(f"  SKIP (already exists): {out_file}")
        summary_rows.append({
            "USGS_site" : USGSsite,
            "status"    : "skipped - already downloaded",
            "n_records" : len(pd.read_csv(out_file))
        })
        continue

    try:
        print(f"Downloading USGS data for site {USGSsite}...")

        # --- Fetch via USGS waterservices RDB ---
        url = (
            "https://waterservices.usgs.gov/nwis/iv/"
            f"?format=rdb&sites={USGSsite}"
            f"&startDT={start_date}&endDT={end_date}"
            f"&parameterCd=00060"
        )
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()

        # Strip comment lines, skip the data-type row (2nd line after header)
        lines    = [l for l in resp.text.splitlines() if not l.startswith("#")]
        rdb_text = "\n".join([lines[0]] + lines[2:])
        df_cd    = pd.read_csv(io.StringIO(rdb_text), sep="\t", dtype=str)

        # ── Dynamic column detection ──────────────────────────────────────
        # Flow column:      contains "00060" but does NOT end in "_cd"
        # Qualifier column: contains "00060" AND ends in "_cd"
        flow_cols = [c for c in df_cd.columns if "00060" in c and not c.endswith("_cd")]
        qual_cols = [c for c in df_cd.columns if "00060" in c and c.endswith("_cd")]

        if not flow_cols:
            print(f"  No 00060 flow column found — columns: {df_cd.columns.tolist()}")
            summary_rows.append({"USGS_site": USGSsite, "status": "no flow column", "n_records": 0})
            continue

        flow_col = flow_cols[0]
        qual_col = qual_cols[0] if qual_cols else None

        # Rename to standard names
        rename_map = {"datetime": "datetime", flow_col: "streamflow_cfs"}
        if qual_col:
            rename_map[qual_col] = "qualifier_code"
        df_cd = df_cd.rename(columns=rename_map)

        df_cd["datetime"] = pd.to_datetime(df_cd["datetime"], utc=True)
        df_cd["streamflow_cfs"] = pd.to_numeric(df_cd["streamflow_cfs"], errors="coerce")
        df_cd["streamflow_m3s"] = df_cd["streamflow_cfs"] * 0.0283168
        df_cd.insert(0, "USGS_site", USGSsite)

        keep_cols = ["USGS_site", "datetime", "streamflow_cfs", "streamflow_m3s"]
        if "qualifier_code" in df_cd.columns:
            keep_cols.append("qualifier_code")
            unique_codes = df_cd["qualifier_code"].dropna().unique()
            print(f"  Qualifier codes found: {unique_codes}")
        else:
            print(f"  No qualifier code column found")

        df_final = df_cd[keep_cols]

        if df_final.empty:
            print(f"  No data returned for {USGSsite}")
            summary_rows.append({"USGS_site": USGSsite, "status": "no data returned", "n_records": 0})
            continue

        df_final.to_csv(out_file, index=False)
        summary_rows.append({"USGS_site": USGSsite, "status": "success", "n_records": len(df_final)})
        print(f"  Saved {len(df_final):,} records → {out_file}")

    except Exception as e:
        print(f"  ERROR for {USGSsite}: {e}")
        summary_rows.append({"USGS_site": USGSsite, "status": f"failed: {e}", "n_records": 0})
        if os.path.exists(out_file):
            os.remove(out_file)

# --- Save summary ---
summary_df   = pd.DataFrame(summary_rows)
summary_path = os.path.join(path_usgs_out, "download_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\nSummary saved → {summary_path}")
print(summary_df)

  ERROR for 07148400: name 'start_date' is not defined


NameError: name 'summary_rows' is not defined

In [8]:
summary_rows = []   # initialize before loop

for USGSsite in USGS_sites:

    out_file = os.path.join(path_usgs_out, f"{USGSsite}_USGS_peak.csv")

    if os.path.exists(out_file):
        print(f"  SKIP (already exists): {out_file}")
        summary_rows.append({
            "USGS_site" : USGSsite,
            "status"    : "skipped - already downloaded",
            "n_records" : len(pd.read_csv(out_file))
        })
        continue

    try:
        print(f"Downloading USGS peak-flow data for site {USGSsite}...")

        # ── Peak-flow RDB endpoint (national, no state prefix needed) ──
        url = (
            "https://nwis.waterdata.usgs.gov/usa/nwis/peak/"
            f"?site_no={USGSsite}&agency_cd=USGS&format=rdb"
        )
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()

        # Strip comment lines (#), keep header + data-type row + data rows
        lines = [l for l in resp.text.splitlines() if not l.startswith("#")]

        if len(lines) < 3:
            print(f"  No peak data returned for {USGSsite}")
            summary_rows.append({"USGS_site": USGSsite, "status": "no data returned", "n_records": 0})
            continue

        # lines[0] = column headers, lines[1] = data-type row (skip), lines[2:] = data
        rdb_text = "\n".join([lines[0]] + lines[2:])
        df_pk = pd.read_csv(io.StringIO(rdb_text), sep="\t", dtype=str)

        # ── Rename to standard columns ──────────────────────────────
        # RDB peak columns:
        #   peak_dt       → date of peak
        #   peak_va       → peak discharge (cfs)
        #   peak_cd       → discharge qualification code  ← key for regulation
        #   gage_ht       → gage height (ft)
        #   gage_ht_cd    → gage height qualification code
        rename_map = {
            "peak_dt"   : "peak_date",
            "peak_va"   : "peak_discharge_cfs",
            "peak_cd"   : "peak_cd",           # e.g. "6" = regulated by reservoir
            "gage_ht"   : "gage_height_ft",
            "gage_ht_cd": "gage_ht_cd",
        }
        df_pk = df_pk.rename(columns={k: v for k, v in rename_map.items() if k in df_pk.columns})

        df_pk["peak_date"]          = pd.to_datetime(df_pk["peak_date"], errors="coerce")
        df_pk["peak_discharge_cfs"] = pd.to_numeric(df_pk["peak_discharge_cfs"], errors="coerce")
        df_pk.insert(0, "USGS_site", USGSsite)

        keep_cols = ["USGS_site", "peak_date", "peak_discharge_cfs", "gage_height_ft", "peak_cd", "gage_ht_cd"]
        keep_cols = [c for c in keep_cols if c in df_pk.columns]   # only keep cols that exist
        df_final  = df_pk[keep_cols].dropna(subset=["peak_discharge_cfs"])

        if df_final.empty:
            print(f"  No valid records for {USGSsite}")
            summary_rows.append({"USGS_site": USGSsite, "status": "no data returned", "n_records": 0})
            continue

        # Show what peak_cd codes are present for this site
        unique_codes = df_final["peak_cd"].dropna().unique() if "peak_cd" in df_final.columns else []
        print(f"  peak_cd codes found: {unique_codes}")

        df_final.to_csv(out_file, index=False)
        summary_rows.append({"USGS_site": USGSsite, "status": "success", "n_records": len(df_final)})
        print(f"  Saved {len(df_final):,} records → {out_file}")

    except Exception as e:
        print(f"  ERROR for {USGSsite}: {e}")
        summary_rows.append({"USGS_site": USGSsite, "status": f"failed: {e}", "n_records": 0})
        if os.path.exists(out_file):
            os.remove(out_file)

# --- Save summary ---
summary_df   = pd.DataFrame(summary_rows)
summary_path = os.path.join(path_usgs_out, "download_summary_peak.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\nSummary saved → {summary_path}")
print(summary_df)

  peak_cd codes found: ['1,2']
  Saved 60 records → Outputs\USGS_ins_data\07148400_USGS_peak.csv
  peak_cd codes found: ['7' '5' '3,5']
  Saved 89 records → Outputs\USGS_ins_data\07152000_USGS_peak.csv
  peak_cd codes found: ['2,7' '7' '5']
  Saved 83 records → Outputs\USGS_ins_data\07153000_USGS_peak.csv
  peak_cd codes found: ['5']
  Saved 59 records → Outputs\USGS_ins_data\07157950_USGS_peak.csv


C:\Users\ronit\AppData\Local\Temp\ipykernel_13352\3289826841.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_pk["peak_date"]          = pd.to_datetime(df_pk["peak_date"], errors="coerce")


  peak_cd codes found: ['5']
  Saved 87 records → Outputs\USGS_ins_data\07158000_USGS_peak.csv
  peak_cd codes found: ['5' '1,5']
  Saved 52 records → Outputs\USGS_ins_data\07159100_USGS_peak.csv
  peak_cd codes found: ['5']
  Saved 33 records → Outputs\USGS_ins_data\07159750_USGS_peak.csv


C:\Users\ronit\AppData\Local\Temp\ipykernel_13352\3289826841.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_pk["peak_date"]          = pd.to_datetime(df_pk["peak_date"], errors="coerce")


  peak_cd codes found: ['7,Bd' '2' '1' '5' '5,R']
  Saved 83 records → Outputs\USGS_ins_data\07160000_USGS_peak.csv
  peak_cd codes found: ['5' '2,5']
  Saved 29 records → Outputs\USGS_ins_data\07160350_USGS_peak.csv
  peak_cd codes found: ['5']
  Saved 68 records → Outputs\USGS_ins_data\07160500_USGS_peak.csv
  peak_cd codes found: ['5']
  Saved 38 records → Outputs\USGS_ins_data\07161450_USGS_peak.csv
  peak_cd codes found: ['5,C']
  Saved 36 records → Outputs\USGS_ins_data\07164600_USGS_peak.csv
  peak_cd codes found: ['5,C' '2,5,C']
  Saved 37 records → Outputs\USGS_ins_data\07165562_USGS_peak.csv
  peak_cd codes found: ['5,C']
  Saved 37 records → Outputs\USGS_ins_data\07165565_USGS_peak.csv
  peak_cd codes found: ['5']
  Saved 52 records → Outputs\USGS_ins_data\07165570_USGS_peak.csv
  peak_cd codes found: ['5' '6']
  Saved 86 records → Outputs\USGS_ins_data\07171000_USGS_peak.csv
  peak_cd codes found: ['5' '2,5']
  Saved 39 records → Outputs\USGS_ins_data\07174400_USGS_peak.csv

...

  peak_cd codes found: ['2,7,Bm' '2,Bm' '5' '6' '1,2,6']
  Saved 76 records → Outputs\USGS_ins_data\07328100_USGS_peak.csv
  peak_cd codes found: ['5' '2,5']
  Saved 36 records → Outputs\USGS_ins_data\07328180_USGS_peak.csv
  peak_cd codes found: ['2' '5' '6' '2,6']
  Saved 87 records → Outputs\USGS_ins_data\07328500_USGS_peak.csv
  peak_cd codes found: ['1,8' '1,2' '1']
  Saved 26 records → Outputs\USGS_ins_data\07329849_USGS_peak.csv
  peak_cd codes found: ['5' '2,5']
  Saved 36 records → Outputs\USGS_ins_data\07329852_USGS_peak.csv
  peak_cd codes found: ['2' '1,2' '5']
  Saved 89 records → Outputs\USGS_ins_data\07332500_USGS_peak.csv
  peak_cd codes found: ['5']
  Saved 88 records → Outputs\USGS_ins_data\07334000_USGS_peak.csv
  peak_cd codes found: ['5' '1,2,5']
  Saved 43 records → Outputs\USGS_ins_data\07335300_USGS_peak.csv
  peak_cd codes found: []
  Saved 60 records → Outputs\USGS_ins_data\07335700_USGS_peak.csv
  peak_cd codes found: ['5' '1,2,5' '1,5']
  Saved 45 records → 

C:\Users\ronit\AppData\Local\Temp\ipykernel_13352\3289826841.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_pk["peak_date"]          = pd.to_datetime(df_pk["peak_date"], errors="coerce")


  peak_cd codes found: ['Bd' '2']
  Saved 65 records → Outputs\USGS_ins_data\07337900_USGS_peak.csv


C:\Users\ronit\AppData\Local\Temp\ipykernel_13352\3289826841.py:55: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_pk["peak_date"]          = pd.to_datetime(df_pk["peak_date"], errors="coerce")


  peak_cd codes found: ['2,Bm' 'Bd' '6']
  Saved 95 records → Outputs\USGS_ins_data\07338500_USGS_peak.csv
  peak_cd codes found: ['2' 'R']
  Saved 32 records → Outputs\USGS_ins_data\07338750_USGS_peak.csv
  peak_cd codes found: ['2,7' '7' '6' '2,6']
  Saved 98 records → Outputs\USGS_ins_data\07339000_USGS_peak.csv

Summary saved → Outputs\USGS_ins_data\download_summary_peak.csv
    USGS_site   status  n_records
0    07148400  success         60
1    07152000  success         89
2    07153000  success         83
3    07157950  success         59
4    07158000  success         87
..        ...      ...        ...
96   07336200  success         53
97   07337900  success         65
98   07338500  success         95
99   07338750  success         32
100  07339000  success         98

[101 rows x 3 columns]
